In [25]:
%pip install pandas sqlalchemy psycopg2-binary scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import sys
print(sys.executable)
import pandas
print("pandas OK")

c:\Users\Kurians Biju\AppData\Local\Python\pythoncore-3.14-64\python.exe
pandas OK


In [27]:
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

password = getpass("Postgres password: ")

url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="credit_risk_db",
)
engine = create_engine(url)

print(pd.read_sql("SELECT COUNT(*) FROM loans", engine))

     count
0  2260668


In [ ]:
query = """
SELECT
    CASE
        WHEN l.loan_status IN ('Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off') THEN 1
        WHEN l.loan_status IN ('Fully Paid', 'Does not meet the credit policy. Status:Fully Paid') THEN 0
    END AS is_default,
    l.loan_amnt, l.term, l.int_rate, l.grade, l.purpose, l.dti, l.delinq_2yrs,
    l.annual_inc, l.verification_status, l.emp_length,
    c.home_ownership,
    l.issue_d
FROM loans l
JOIN customers c ON l.customer_id = c.customer_id
WHERE l.loan_status IN (
        'Charged Off',
        'Default',
        'Does not meet the credit policy. Status:Charged Off',
        'Fully Paid',
        'Does not meet the credit policy. Status:Fully Paid'
      )
  AND l.issue_d < '2016-01-01';
"""

df = pd.read_sql(query, engine)

print(df.shape)
print(df["is_default"].mean())
df.head()

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 825575 entries, 0 to 825574
Data columns (total 13 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   is_default           825575 non-null  int64  
 1   loan_amnt            825575 non-null  float64
 2   term                 825575 non-null  str    
 3   int_rate             825575 non-null  float64
 4   grade                825575 non-null  str    
 5   purpose              825575 non-null  str    
 6   dti                  825573 non-null  float64
 7   delinq_2yrs          825546 non-null  float64
 8   annual_inc           825571 non-null  float64
 9   verification_status  825575 non-null  str    
 10  emp_length           783104 non-null  str    
 11  home_ownership       825575 non-null  str    
 12  issue_d              825575 non-null  object 
dtypes: float64(5), int64(1), object(1), str(6)
memory usage: 81.9+ MB


In [ ]:
num_cols = ["loan_amnt", "int_rate", "dti", "annual_inc"]
df[num_cols] = df[num_cols].astype(float)
df["issue_d"] = pd.to_datetime(df["issue_d"])

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 825575 entries, 0 to 825574
Data columns (total 13 columns):
 #   Column               Non-Null Count   Dtype        
---  ------               --------------   -----        
 0   is_default           825575 non-null  int64        
 1   loan_amnt            825575 non-null  float64      
 2   term                 825575 non-null  str          
 3   int_rate             825575 non-null  float64      
 4   grade                825575 non-null  str          
 5   purpose              825575 non-null  str          
 6   dti                  825573 non-null  float64      
 7   delinq_2yrs          825546 non-null  float64      
 8   annual_inc           825571 non-null  float64      
 9   verification_status  825575 non-null  str          
 10  emp_length           783104 non-null  str          
 11  home_ownership       825575 non-null  str          
 12  issue_d              825575 non-null  datetime64[s]
dtypes: datetime64[s](1), float64(5), int64(1

In [ ]:
df["issue_d"] = pd.to_datetime(df["issue_d"])
df = df.dropna(subset=["dti", "delinq_2yrs", "annual_inc"])
print(df.shape)

(825544, 13)


In [ ]:
df.groupby(df["emp_length"].isna())["is_default"].agg(["mean", "count"])

,mean,count
emp_length,,
False,0.181811,783075
True,0.236690,42469


In [ ]:
# Flag: was emp_length missing? (1 = yes)
df["emp_length_missing"] = df["emp_length"].isna().astype(int)

# Pull the number out of text like '10+ years' or '3 years'
df["emp_years"] = df["emp_length"].str.extract(r"(\d+)")[0].astype(float)

# '< 1 year' contains a '1', but it means less than one year, so set it to 0
df.loc[df["emp_length"] == "< 1 year", "emp_years"] = 0

# Fill the blanks with the median so those rows aren't treated as '< 1 year'
df["emp_years"] = df["emp_years"].fillna(df["emp_years"].median())

# term: ' 36 months' -> 36
df["term_months"] = df["term"].str.extract(r"(\d+)")[0].astype(int)

In [ ]:
print(df["emp_years"].value_counts().sort_index())
print(df["term_months"].value_counts())

emp_years
0.0      66117
1.0      53625
2.0      73841
3.0      65484
4.0      49168
5.0      52251
6.0      82852
7.0      41762
8.0      40872
9.0      32020
10.0    267552
Name: count, dtype: int64
term_months
36    620818
60    204726
Name: count, dtype: int64


In [ ]:
df = df.drop(columns=["emp_length", "term"])   # replaced by numeric versions

cat_cols = ["grade", "purpose", "home_ownership", "verification_status"]
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
print(df.shape)
df.head()

(825544, 36)


,is_default,loan_amnt,int_rate,dti,delinq_2yrs,annual_inc,issue_d,emp_length_missing,emp_years,term_months,...,purpose_small_business,purpose_vacation,purpose_wedding,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,verification_status_Source Verified,verification_status_Verified
0,1,6000.0,9.99,24.27,0.0,50000.0,2015-06-01,0,7.0,36,...,0,0,0,0,0,0,0,1,1,0
1,1,20575.0,19.19,26.59,0.0,49000.0,2015-06-01,0,4.0,60,...,0,0,0,0,0,0,0,1,1,0
2,0,15000.0,9.17,24.92,0.0,64000.0,2015-06-01,0,10.0,36,...,0,0,0,1,0,0,0,0,0,0
3,0,20000.0,15.61,36.85,0.0,89500.0,2015-06-01,0,2.0,60,...,0,0,0,1,0,0,0,0,0,1
4,0,6450.0,14.65,26.06,0.0,17000.0,2015-06-01,0,3.0,36,...,0,0,0,0,0,0,1,0,0,0


In [ ]:
home_cols = [c for c in df.columns if c.startswith("home_ownership_")]
print((df[home_cols].sum(axis=1) == 0).sum())

3


In [ ]:
df = df[df[home_cols].sum(axis=1) > 0]
df = df.drop(columns=["home_ownership_MORTGAGE"])

In [ ]:
train = df[df["issue_d"] < "2015-01-01"]
test  = df[df["issue_d"] >= "2015-01-01"]

feature_cols = [c for c in df.columns if c not in ["is_default", "issue_d"]]

X_train, y_train = train[feature_cols], train["is_default"]
X_test,  y_test  = test[feature_cols],  test["is_default"]

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(452133, 33) (373408, 33)
0.1706356315508932 0.2015864684206016


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

num_cols = ["loan_amnt", "int_rate", "dti", "delinq_2yrs",
            "annual_inc", "emp_years", "term_months"]

X_train = X_train.copy()
X_test = X_test.copy()

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pd_scores = model.predict_proba(X_test)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, pd_scores))

coefs = pd.Series(model.coef_[0], index=X_train.columns).sort_values()
print(coefs)

ROC-AUC: 0.7258281052834267
purpose_wedding                       -0.472820
annual_inc                            -0.210862
purpose_major_purchase                -0.065880
purpose_credit_card                   -0.062936
purpose_vacation                      -0.059115
verification_status_Verified          -0.052872
purpose_other                         -0.031293
purpose_house                         -0.020917
emp_years                             -0.010163
purpose_debt_consolidation             0.003146
home_ownership_NONE                    0.010848
delinq_2yrs                            0.014609
purpose_home_improvement               0.025582
purpose_renewable_energy               0.039045
purpose_medical                        0.062168
loan_amnt                              0.068284
purpose_moving                         0.081333
home_ownership_OWN                     0.087024
verification_status_Source Verified    0.098916
dti                                    0.124788
int_rate    

In [ ]:
ver_cols = ["verification_status_Verified", "verification_status_Source Verified"]
grade_cols = [c for c in X_train.columns if c.startswith("grade_")]

def verified_coef(cols):
    m = LogisticRegression(max_iter=1000)
    m.fit(X_train[cols], y_train)
    return pd.Series(m.coef_[0], index=cols)[ver_cols]

print(verified_coef(ver_cols))                           # verification alone
print(verified_coef(ver_cols + grade_cols))              # + grade
print(verified_coef(list(X_train.columns)))              # everything

verification_status_Verified           0.456351
verification_status_Source Verified    0.452732
dtype: float64
verification_status_Verified           0.146264
verification_status_Source Verified    0.240733
dtype: float64
verification_status_Verified          -0.052872
verification_status_Source Verified    0.098916
dtype: float64


In [ ]:
print(pd_scores.mean(), y_test.mean())

0.1663739541256975 0.2015864684206016


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

for t in [0.5, 0.3, 0.2]:
    pred = (pd_scores >= t).astype(int)
    print(t,
          "precision:", round(precision_score(y_test, pred), 3),
          "recall:", round(recall_score(y_test, pred), 3),
          "F1:", round(f1_score(y_test, pred), 3))

0.5 precision: 0.626 recall: 0.012 F1: 0.024
0.3 precision: 0.472 recall: 0.287 F1: 0.357
0.2 precision: 0.369 recall: 0.556 F1: 0.443


In [ ]:
from sklearn.metrics import confusion_matrix

cost_fn = 5   # cost of approving a loan that defaults
cost_fp = 1   # cost of rejecting a loan that would have been repaid

for t in [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    pred = (pd_scores >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(t, "missed defaults:", fn, "good rejected:", fp, "cost:", cost_fn * fn + cost_fp * fp)

0.1 missed defaults: 8389 good rejected: 190224 cost: 232169
0.15 missed defaults: 20300 good rejected: 120941 cost: 222441
0.2 missed defaults: 33453 good rejected: 71660 cost: 238925
0.25 missed defaults: 44699 good rejected: 41467 cost: 264962
0.3 missed defaults: 53669 good rejected: 24126 cost: 292471
0.4 missed defaults: 67736 good rejected: 6003 cost: 344683
0.5 missed defaults: 74337 good rejected: 561 cost: 372246


In [ ]:
cost_fn = 5985
interest = 2691
thresholds = [i / 100 for i in range(5, 61, 5)]

for share in [1.0, 0.5, 0.25]:
    cost_fp = interest * share
    results = []
    for t in thresholds:
        pred = (pd_scores >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
        results.append((cost_fn * fn + cost_fp * fp, t, round(pred.mean(), 3)))
    best = min(results)
    print(f"FP cost = {share:.0%} of gross interest -> best threshold {best[1]}, "
          f"cost {best[0]:,.0f}, share of applicants flagged {best[2]}")

FP cost = 100% of gross interest -> best threshold 0.25, cost 379,111,212, share of applicants flagged 0.193
FP cost = 50% of gross interest -> best threshold 0.15, cost 284,221,616, share of applicants flagged 0.471
FP cost = 25% of gross interest -> best threshold 0.1, cost 178,181,361, share of applicants flagged 0.689


In [ ]:
drop = [c for c in X_train.columns if c.startswith("grade_")] + ["int_rate"]
keep = [c for c in X_train.columns if c not in drop]

m2 = LogisticRegression(max_iter=1000)
m2.fit(X_train[keep], y_train)
scores2 = m2.predict_proba(X_test[keep])[:, 1]

print("AUC without grade & int_rate:", roc_auc_score(y_test, scores2))
print(pd.Series(m2.coef_[0], index=keep).sort_values())

AUC without grade & int_rate: 0.6863634528245437
annual_inc                            -0.261030
purpose_wedding                       -0.032653
emp_years                             -0.007139
purpose_credit_card                    0.039645
delinq_2yrs                            0.052594
home_ownership_NONE                    0.057011
verification_status_Verified           0.075330
loan_amnt                              0.080897
home_ownership_OWN                     0.149105
purpose_major_purchase                 0.152201
verification_status_Source Verified    0.157715
dti                                    0.170691
purpose_debt_consolidation             0.239924
purpose_home_improvement               0.283995
purpose_renewable_energy               0.304758
home_ownership_RENT                    0.332701
emp_length_missing                     0.360649
purpose_house                          0.381223
term_months                            0.412083
purpose_vacation                       